<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/media/logo_dataprojectlab.png" width="220"/>
</div>

# MarketPulse — Notebook 2 — Data Cleaning & Feature Engineering
### 📝 VERSION APPRENANT

> Objectif : Corriger les 6 anomalies, recalculer les métriques, construire les features ML.
> Granularité finale : 1 ligne = 1 campagne.

---
## 0. Setup

In [ ]:
!pip install jupysql==0.11.1 duckdb-engine --quiet

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os, sys

pd.set_option('display.max_columns', 50)
COLORS = {'primary':'#534AB7','secondary':'#1D9E75','warning':'#EF9F27','danger':'#E24B4A','neutral':'#888780','light':'#EEEDFE'}
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#F9F9F8','axes.grid':True,'grid.alpha':0.35,'font.size':11})

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive; drive.mount('/content/drive')
    SAVE_PATH = '/content/drive/MyDrive/DataProjectLab/projects/marketpulse/'
else:
    SAVE_PATH = './outputs/'
os.makedirs(SAVE_PATH, exist_ok=True)
BASE_URL = 'https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/marketpulse/data/'
print(f'Configuration OK — {SAVE_PATH}')


---
## Étape 1 — Chargement depuis zéro

### MÉTHODE
Recharger les CSV depuis GitHub. `parse_dates` sur les colonnes dates.

In [ ]:
# 📝 TODO : Charger les 7 CSV avec parse_dates
# 💡 Indice 1 : pd.read_csv(BASE_URL + 'performances_daily.csv', parse_dates=['date_perf'])
# 💡 Indice 2 : Tables : performances_daily, campagnes, clients_agence, audiences, creatifs, conversions, contacts_crm

df_perf = ...
df_camp = ...
df_cli  = ...
df_aud  = ...
df_crea = ...
df_conv = ...
df_cont = ...
print('Chargement OK ✅')


---
## Étape 2 — Sauvegarde brute

### MÉTHODE — Nettoyage non destructif
Sauvegarde avant toute modification. Ne jamais modifier les sources directement.

In [ ]:
# 📝 TODO : Sauvegarder les 3 tables à modifier (copy)
# 💡 Indice 1 : df_perf_raw = df_perf.copy()
# 💡 Indice 2 : Faire de même pour camp et conv

df_perf_raw = ...
df_camp_raw = ...
df_conv_raw = ...
print('Bruts sauvegardés ✅')


---
## Étape 3 — Nettoyage `performances_daily`

### MÉTHODE
Ordre : 1) impossible (clics>impressions) → 2) signe incorrect (revenu<0) → 3) aberrant (ROAS>50) → 4) recalcul universel

### 3.1 — Clics > impressions (4 lignes)

In [ ]:
# 📝 TODO : Corriger les lignes avec clics > impressions
# 💡 Indice 1 : mask_clics = df_perf['clics'] > df_perf['impressions']
# 💡 Indice 2 : Correction : impressions = max(impressions, clics)
# 💡 Indice 3 : df_perf.loc[mask, 'impressions'] = df_perf.loc[mask, ['impressions','clics']].max(axis=1)

mask_clics = ...
print(f'Avant : {mask_clics.sum()} lignes')
# Correction
print(f'Après : {(df_perf["clics"] > df_perf["impressions"]).sum()} lignes')


### 3.2 — Revenus négatifs → NULL

In [ ]:
# 📝 TODO : Remplacer les revenus négatifs par NULL (pas 0)
# 💡 Indice 1 : mask_rev = df_perf['revenu_genere_fcfa'] < 0
# 💡 Indice 2 : df_perf.loc[mask_rev, 'revenu_genere_fcfa'] = np.nan

mask_rev = ...
# Correction
print(f'Revenus NULL : {df_perf["revenu_genere_fcfa"].isna().sum()}')


### 3.3 — ROAS aberrant > 50

In [ ]:
# 📝 TODO : Recalculer les ROAS > 50 depuis revenu/dépenses
# 💡 Indice 1 : mask_roas = df_perf['roas'] > 50
# 💡 Indice 2 : Vérifier chaque ligne : ROAS déclaré vs calculé manuellement
# 💡 Indice 3 : df_perf.loc[mask, 'roas'] = revenu_genere / depenses_fcfa

mask_roas = ...
# Correction
print(f'ROAS > 50 restants : {(df_perf["roas"] > 50).sum()}')


### 3.4 — Recalcul universel CTR, ROAS, CPA, taux_conversion

### MÉTHODE
Recalculer **toute** la colonne depuis les sources primitives — pas seulement les lignes aberrantes.

In [ ]:
# 📝 TODO : Créer 4 colonnes recalculées : ctr_recalc, roas_recalc, cpa_recalc, taux_conversion_recalc
# 💡 Indice 1 : ctr_recalc = clics / impressions.replace(0, np.nan)
# 💡 Indice 2 : roas_recalc = revenu_genere_fcfa / depenses_fcfa.replace(0, np.nan)
# 💡 Indice 3 : cpa_recalc = depenses / conversions (si conversions > 0, sinon NaN)

df_perf['ctr_recalc']             = ...
df_perf['roas_recalc']            = ...
df_perf['cpa_recalc']             = ...
df_perf['taux_conversion_recalc'] = ...
print(f'CTR recalc moyen : {df_perf["ctr_recalc"].mean()*100:.2f}%')


### 🧠 Tes observations

> - Le CTR moyen recalculé est-il proche de 2.86% (référence NB1) ?
> - Pourquoi recalculer toute la colonne est-il plus robuste que corriger seulement les 38 aberrants ?


---
## Étape 4 — Nettoyage `campagnes` et `conversions`

In [ ]:
# 📝 TODO : Créer le flag depassement_budget et le ratio_depense_budget
# 💡 Indice 1 : depassement_budget = (budget_depense > budget_alloue).astype(int)
# 💡 Indice 2 : ratio = budget_depense / budget_alloue.replace(0, np.nan)

df_camp['depassement_budget']   = ...
df_camp['ratio_depense_budget'] = ...
print(f'Dépassements : {df_camp["depassement_budget"].sum()}')


In [ ]:
# 📝 TODO : Imputer les valeurs d'achat NULL par médiane du canal_source
# 💡 Indice 1 : mask = (type_conversion == 'Achat') & valeur_fcfa.isna()
# 💡 Indice 2 : medianes = df_conv[achat OK].groupby('canal_source')['valeur_fcfa'].median()
# 💡 Indice 3 : Merger les médianes puis df_conv.loc[mask, 'valeur_fcfa'] = médiane correspondante

mask_achats_null = ...
medianes_conv = ...
# Merger et imputer
print(f'Achats NULL restants : {((df_conv["type_conversion"]=="Achat") & df_conv["valeur_fcfa"].isna()).sum()}')


### 🧠 Tes observations

> - La médiane d'achat varie-t-elle beaucoup entre Email et Facebook ? Quelle logique métier explique cela ?
> - Pourquoi ne supprime-t-on pas les 3 campagnes avec dépassement budget ?


---
## Étape 5 — Feature Engineering

### MÉTHODE
Table analytique = 354 lignes (1 par campagne). Features disponibles à J3 uniquement.

⚠️ **Leakage à éviter :** `roas_total`, `ctr_total`, `conversions_total` — calculés sur toute la campagne, inconnus à J3.

### 5.1 — Features temporelles

In [ ]:
# 📝 TODO : Créer les features temporelles dans df_analytics
# 💡 Indice 1 : duree_jours = (date_fin - date_debut).dt.days
# 💡 Indice 2 : est_lancement_weekend = (date_debut.dt.dayofweek >= 5).astype(int)
# 💡 Indice 3 : mois_sin = np.sin(2 * np.pi * mois / 12)  # encodage cyclique

df_analytics = df_camp[['campagne_id','client_id','canal','objectif_campagne','date_debut','date_fin','budget_alloue_fcfa','budget_depense_fcfa','pays_cible','audience_id','statut','ratio_depense_budget','depassement_budget','campagne_sous_performe']].copy()
df_analytics['duree_jours']           = ...
df_analytics['est_lancement_weekend'] = ...
df_analytics['mois_sin']              = ...
df_analytics['mois_cos']              = ...
df_analytics['annee_lancement']       = ...


### 5.2 — Signaux précoces J1-J3 ← features clés pour le ML

### MÉTHODE
Filtrer `performances_daily` avec `jour_campagne <= 3`. Exclure les anomalies. Ces features simulent ce que le modèle voit en production à J3.

In [ ]:
# 📝 TODO : Calculer les KPIs des 3 premiers jours de chaque campagne
# 💡 Indice 1 : df_perf_clean = df_perf[(clics<=impressions) & (revenu>=0) & (roas<=50) & (ctr_recalc<=0.20)]
# 💡 Indice 2 : df_j3 = df_perf_clean[jour_campagne<=3].groupby('campagne_id').agg(...)
# 💡 Indice 3 : ctr_j3 = clics_j3 / impressions_j3  |  roas_j3 = revenu_j3 / depenses_j3

df_perf_clean = df_perf[(df_perf['clics']<=df_perf['impressions'])&(df_perf['revenu_genere_fcfa']>=0)&(df_perf['roas']<=50)&(df_perf['ctr_recalc']<=0.20)].copy()
df_j3 = ...  # agrégation groupby campagne_id sur jour_campagne <= 3
df_j3['ctr_j3']  = ...
df_j3['roas_j3'] = ...
df_j3['taux_conv_j3'] = ...
print(f'ROAS J3 moyen : {df_j3["roas_j3"].mean():.3f}')


In [ ]:
# 📝 TODO : Calculer variation CTR J1→J3 et ratio dépenses J3/budget
# 💡 Indice 1 : df_j1 = df_perf_clean[jour_campagne==1].groupby('campagne_id').agg(ctr_j1=..., roas_j1=...)
# 💡 Indice 2 : variation_ctr_j1_j3 = ctr_j3 - ctr_j1
# 💡 Indice 3 : ratio_dep_j3_vs_budget = depenses_j3 / budget_alloue_fcfa

df_j1 = ...
df_j3 = df_j3.merge(df_j1, on='campagne_id', how='left')
df_j3['variation_ctr_j1_j3']  = ...
df_j3['variation_roas_j1_j3'] = ...
df_j3['ratio_dep_j3_vs_budget'] = ...
print(f'Campagnes CTR décroissant dès J3 : {(df_j3["variation_ctr_j1_j3"] < 0).sum()}')


### 🧠 Tes observations

> - Quel % de campagnes ont un ROAS J3 < 1.5 ? Ces campagnes sont-elles majoritairement sous-performantes ?
> - Visualise la distribution de ROAS_J3 par classe (campagne_sous_performe). Y a-t-il une séparation ?


### 5.3 — Performance totale (NB3 + leakage ML)

In [ ]:
# 📝 TODO : Agréger la performance totale de chaque campagne
# 💡 Indice 1 : df_perf_total = df_perf_clean.groupby('campagne_id').agg(impressions_total=..., clics_total=..., ...)
# 💡 Indice 2 : KPIs : ctr_total, roas_total, cpa_total, taux_conv_total, cpc_total

df_perf_total = df_perf_clean.groupby('campagne_id').agg(
    impressions_total=('impressions','sum'),
    clics_total=('clics','sum'),
    depenses_total=('depenses_fcfa','sum'),
    conversions_total=('conversions','sum'),
    revenu_total=('revenu_genere_fcfa','sum')
)

df_perf_total['roas_total'] = ...
df_perf_total['ctr_total']  = ...


### 5.4 — Contexte historique

### MÉTHODE
Taux historiques par canal/client/objectif = prédicteurs *a priori* connus avant le lancement.

In [ ]:
# 📝 TODO : Calculer taux_sous_perf_canal, taux_sous_perf_client, taux_sous_perf_objectif
# 💡 Indice 1 : df_camp.groupby('canal')['campagne_sous_performe'].mean().reset_index()
# 💡 Indice 2 : Renommer en taux_sous_perf_canal
# 💡 Indice 3 : Merger dans df_analytics on='canal'

taux_canal    = ...
taux_client   = ...
taux_objectif = ...
df_analytics = df_analytics.merge(taux_canal,    on='canal',             how='left')
df_analytics = df_analytics.merge(taux_client,   on='client_id',         how='left')
df_analytics = df_analytics.merge(taux_objectif, on='objectif_campagne', how='left')


### 5.5 — Audience + indice de saturation

In [ ]:
# 📝 TODO : Enrichir depuis audiences + calculer indice_saturation_j3
# 💡 Indice 1 : merge df_aud sur audience_id pour taille_estimee, type_audience
# 💡 Indice 2 : indice_saturation_j3 = depenses_j3 / taille_estimee * 1000
# 💡 Indice 3 : Plus l'indice est élevé, plus l'audience est saturée

df_analytics = df_analytics.merge(df_aud[['audience_id','taille_estimee','type_audience','tranche_age','genre']], on='audience_id', how='left')
df_analytics = df_analytics.merge(df_j3[['campagne_id','depenses_j3']], on='campagne_id', how='left')
df_analytics['indice_saturation_j3'] = ...


### 5.6 — Encodages ordinaux

In [ ]:
# 📝 TODO : Encoder canal, objectif et type_audience en numériques
# 💡 Indice 1 : canal_map = {'Email':0, 'SMS':1, 'Google':2, 'Facebook':3}
# 💡 Indice 2 : objectif_map = {'Notoriété':0,'Trafic':1,'Leads':2,'Retargeting':3,'Conversion':4}
# 💡 Indice 3 : df['col_num'] = df['col'].map(map_dict)

canal_map = ...
objectif_map = ...
aud_map = {'Froide':0,'Lookalike':1,'Retargeting':2,'CRM':3}
df_analytics['canal_num']         = ...
df_analytics['objectif_num']      = ...
df_analytics['type_audience_num'] = ...


---
## Étape 6 — Assemblage final

### MÉTHODE
Merge dans l'ordre : métadonnées → performance totale → signaux J3 → contexte. Imputer les NaN résiduels par médiane du canal.

In [ ]:
# 📝 TODO : Assembler la table analytique en mergant toutes les parties
# 💡 Indice 1 : df_analytics.merge(df_perf_total[[cols...]], on='campagne_id', how='left')
# 💡 Indice 2 : df_analytics.merge(df_j3[[cols_j3...]], on='campagne_id', how='left')
# 💡 Indice 3 : Imputer NaN features J3 par médiane du canal

# Merge performance totale
# ...
# Merge signaux J3
# ...
# Imputation
print(f'Table : {df_analytics.shape}')
print(f'Nulls : {df_analytics.isnull().sum().sum()}')


---
## Étape 7 — Validation et export

In [ ]:
# 📝 TODO : Revalider campagne_sous_performe et créer un boxplot ROAS J3 par classe
# 💡 Indice 1 : sous_perf_rev = ((roas_total < 1.5) | (taux_conv_total < 0.01)).astype(int)
# 💡 Indice 2 : Comparer avec campagne_sous_performe — si diff > 0, mettre à jour
# 💡 Indice 3 : df_analytics.boxplot(column='roas_j3', by='campagne_sous_performe')

# Revalidation
sous_perf_revalidee = ...
diff = ...
print(f'Écarts : {diff}')
# Boxplot
fig, ax = plt.subplots(figsize=(8,5))
# ...


### 🧠 Tes observations

> - Les distributions ROAS J3 sont-elles séparées entre les deux classes ?
> - Le taux de positifs (26.3%) est-il équilibré pour le ML ? A-t-on besoin de SMOTE ?


In [ ]:
# 📝 TODO : Sélectionner les colonnes et exporter marketpulse_analytics.csv
# 💡 Indice 1 : Exclure les leakage : roas_total, ctr_total, conversions_total...
# 💡 Indice 2 : df_export.to_csv(f'{SAVE_PATH}marketpulse_analytics.csv', index=False)
# 💡 Indice 3 : Vérifier : 354 lignes, 0 null sur features J3

COLS_EXPORT = [
    'campagne_id','client_id','canal','objectif_campagne','statut',
    'date_debut','date_fin','duree_jours','mois_sin','mois_cos',
    'est_lancement_weekend','budget_alloue_fcfa','ratio_depense_budget',
    'ctr_j3','roas_j3','taux_conv_j3','variation_ctr_j1_j3','variation_roas_j1_j3',
    'ratio_dep_j3_vs_budget','taux_sous_perf_canal','taux_sous_perf_client',
    'taille_estimee','indice_saturation_j3','canal_num','objectif_num',
    'roas_total','ctr_total','cpa_total','campagne_sous_performe',
]
df_export = df_analytics[[c for c in COLS_EXPORT if c in df_analytics.columns]]
# Exporter...
print(f'Exporté : {df_export.shape}')


---
## Bilan

### Corrections à documenter
| Anomalie | Nb | Méthode |
|---|---|---|
| Clics > impressions | ? | ✍️ |
| Revenus négatifs | ? | ✍️ |
| ROAS > 50 | ? | ✍️ |
| Recalcul CTR/ROAS | tous | ✍️ |
| Budget dépassé | ? | ✍️ |
| Valeur achat NULL | ? | ✍️ |


---

**DataProjectLab** — apprendre la data sur des cas concrets, structurés et orientés métier.